# 06 · Guardrails and approval

Notebook 03 ended with the agent refunding $250 for the late espresso machine with nobody's approval.
Policy P5 says any refund over $100 needs a supervisor, the model had read it, and it refunded anyway. A
rule in a prompt is advice; a **guardrail** is a check in code that the model cannot skip.

**You will learn**

- the guardrails ShipFast already has, and the ones CrewAI adds
- how a CrewAI Flow pauses for a person with `@human_feedback`
- how a paused flow is saved and later resumed with a decision
- why, in CrewAI, the refund has to become its own flow step

In [1]:
import json

from crew_agents.approval import ShipFastApprovalFlow, resume  # import crew_agents first: telemetry off
from crewai.flow.async_feedback import HumanFeedbackPending

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast

verify_model()
LATE = "Order A1003 is more than a week late. Please cancel it and refund me."


def ledger():
    return [(r["order_id"], r["amount"]) for r in shipfast.refunds()] or "no refund"

## Guardrails so far

| Guardrail | Where it lives | What it stops |
|---|---|---|
| `issue_refund` refuses more than the order total, or than what is left after earlier refunds | the shared tool | a made-up amount (02), a double refund (04) |
| `max_iter` | the agent | a run that never stops (03) |
| the ledger check | ShipFast's test runner | a reply or result that claims a refund that never happened |
| a `Task` guardrail | CrewAI: `Task(guardrail=...)` | a task output that fails your own check; CrewAI sends it back for another attempt |
| **approval for refunds over $100** | **a Flow step: this notebook** | money moving without a person's yes |

The task guardrail is CrewAI's built-in output check: a function that receives the task's output and
returns `(True, output)` or `(False, "what to fix")`, and CrewAI retries the task with the feedback. It
checks what the agent *says*. Approval is different: the run must **stop and wait** for a person, possibly
for hours, before any money moves.

## Where CrewAI can pause

LangGraph can pause in the middle of a tool call. CrewAI pauses **between steps of a Flow**: a method
decorated with `@human_feedback` runs, then the flow asks a person for feedback before going on. So the
refund cannot stay a tool call buried inside the agent's loop. ShipFast's CrewAI gate splits it:

1. The agent gets a **gated** `issue_refund`. Up to $100 it runs at once. Above that it does not run: the
   request is recorded in the flow's state as pending, and the agent is told a supervisor must approve.
2. A router sends a flow with a pending refund to a `@human_feedback` step, which pauses the flow.
3. On **approve**, code issues exactly the pending refund. On **reject**, the agent is told and must
   escalate to a human.

Here is all of it, from `crew_agents/approval.py`:

In [2]:
import inspect

import crew_agents.approval

print(inspect.getsource(crew_agents.approval))

"""ShipFast's approval gate (SPEC.md section 3, policy P5) as a CrewAI Flow with `@human_feedback`.

CrewAI pauses between flow steps, not inside a tool call, so the gate has two parts:

1. The specialist agent gets a gated `issue_refund`. Refunds up to $100.00 run at once; a larger one is
   not run, only recorded as pending in the flow's state, and the agent is told it awaits a supervisor.
2. A router sends a flow with a pending refund to a `@human_feedback` step, which pauses the flow
   (`kickoff` returns `HumanFeedbackPending`) and saves its state with `@persist`.
3. `resume(flow_id, "approve")` restores the flow from that saved state and issues exactly the pending
   refund, in code; `resume(flow_id, "reject")` tells the agent, which must escalate to a human. CrewAI
   only resumes from saved state, so persistence is required, here a SQLite file inside the tutorial.

The feedback is mapped to "approve" or "reject" by the local model: CrewAI's default for that mapping is
an OpenAI

Three details in that code. `SupervisorQueue` is a feedback **provider** that never blocks: it raises
`HumanFeedbackPending`, which CrewAI turns into a pause instead of a console prompt (a notebook has no
console). `@persist` saves the flow's state to SQLite, because CrewAI resumes only from saved state. And
`llm=FEEDBACK_LLM`: CrewAI maps the person's free-text reply onto `approve` or `reject` with a model, and
its default for that is an OpenAI model, so the local one is set.

## Under $100: no pause

The lost running shoes (A1002) are a $45 refund. The gate lets it straight through.

In [3]:
shipfast.reset()
flow = ShipFastApprovalFlow()
out = await flow.kickoff_async(inputs={"message": "My order A1002 never arrived and tracking says it's lost. I want a refund."})
print("paused:", isinstance(out, HumanFeedbackPending))
print("ledger:", ledger())
print("reply: ", flow.state.reply.strip()[:160])

paused: False
ledger: [('A1002', 45.0)]
reply:  Your refund for order A1002 has been issued successfully. The total amount of $45.00 will be refunded to your original payment method. Thank you for your patien


## Over $100: the flow stops

The espresso machine (A1003) is $250. `kickoff` returns a `HumanFeedbackPending` instead of a result.
Nothing has been refunded; the request waits in the flow's saved state, under the flow's id.

In [4]:
shipfast.reset()
flow = ShipFastApprovalFlow()
paused = await flow.kickoff_async(inputs={"message": LATE})

print("paused:", isinstance(paused, HumanFeedbackPending))
print("flow id:", paused.context.flow_id)
print("pending refund:", flow.state.pending_refund)
print("ledger while paused:", ledger())
print("told the customer:", flow.state.reply.strip()[:200])

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastApprovalFlow                                                                                     │
│  ID: b85cad71-12ae-4f85-ba18-74a78e3e52ba                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

paused: True
flow id: b85cad71-12ae-4f85-ba18-74a78e3e52ba
pending refund: {'order_id': 'A1003', 'amount': 250.0, 'reason': 'The order is delayed beyond the ETA and the customer is requesting a refund.'}
ledger while paused: no refund
told the customer: A supervisor must approve this refund. You will hear back shortly. We apologize for the inconvenience caused by the delay.


## The supervisor approves

`resume` restores the flow **from its saved state**, by id, and continues it with the decision. That is
deliberate: the supervisor's answer could come hours later, from another process. Code then issues
exactly the amount the supervisor saw; the model does not get another chance to change it.

In [5]:
approved = await resume(paused.context.flow_id, "approve")
print("ledger:", ledger())
print("reply: ", approved.state.reply)

ledger: [('A1003', 250.0)]
reply:  Good news: your refund of $250.00 for order A1003 has been approved and issued.


## The supervisor rejects

Same message, new flow, and this time `reject`. No refund is issued; the agent is told, and policy P5
says a rejected refund goes to a human agent.

In [6]:
shipfast.reset()
flow = ShipFastApprovalFlow()
paused = await flow.kickoff_async(inputs={"message": LATE})
rejected = await resume(paused.context.flow_id, "reject")
print("ledger: ", ledger())
print("tickets:", [t["ticket_id"] for t in shipfast.tickets()])
print("tools:  ", shipfast.tool_calls())
print("reply:  ", rejected.state.reply.strip()[:200])

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastApprovalFlow                                                                                     │
│  ID: 61cfa11b-31fe-4cab-b773-76296df4d628                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ledger:  no refund
tickets: ['TCK-A1003']
tools:   ['lookup_order', 'track_shipment', 'search_policy', 'escalate_to_human']
reply:   A specialist has been assigned to your case, ticket TCK-A1003. We will follow up with you shortly. We apologize for the inconvenience caused by the delay.


The refund cannot happen without `approve`, because the gated tool never calls the real one above
$100, and after approval it is code, not the model, that issues the refund. The escalation after a
rejection *is* left to the model, and ShipFast's tests check it: scenario S3b expects `escalated`.

## How the others do this

- **LangGraph** pauses inside the tool call itself: middleware wraps every tool call and calls
  `interrupt()` before a refund over $100; `Command(resume="approve")` re-runs the same call. The agent
  stays one loop, and a checkpointer (in memory is enough within one process) holds the paused state.
- **Strands** can interrupt from a hook on `BeforeToolCallEvent`, which fires before each tool call;
  the hook checks the refund amount and pauses, closer to LangGraph's shape than to a Flow step.

## Takeaways

- A prompt asks; a guardrail enforces. Put money-moving checks in code.
- CrewAI pauses between Flow steps with `@human_feedback`; a non-blocking provider turns that into a pause
  the caller resumes later.
- CrewAI resumes only from saved state, so the flow needs `@persist`.
- Because the pause sits between steps, the refund itself becomes a step, and code issues the approved
  amount.
- Next: **07 · Tracing and eval**, measuring runs the same way in all three frameworks.